# Trace Collection (Observability MCP)

Implement and visualize the trace collector that records every operation in the research harness.

## Learning Objectives
- Understand trace event structure
- Use the TraceCollector to record operations
- Persist and query traces from PostgreSQL
- Understand how traces feed into the harness Reflect phase

In [ ]:
import os
import sys
import httpx
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
env_path = os.path.join('..', '.env')
load_dotenv(env_path, override=True)

_VERIFY_SSL = os.getenv("VERIFY_SSL", "true").lower() not in ("false", "0", "no")
_http_client = None if _VERIFY_SSL else httpx.Client(verify=False, timeout=httpx.Timeout(300.0))

from harness.trace import TraceEvent, TraceCollector
print("Trace module loaded")

## 1. TraceEvent Structure

Every operation produces a `TraceEvent` with:
- **session_id**: Links to the research session
- **iteration**: Which pass this occurred in
- **layer**: context | tool | execution | verification
- **operation**: Specific operation name
- **tokens_used / latency_ms**: Cost metrics
- **success / failure_category**: Outcome tracking

In [ ]:
# Create a trace collector and record some events
collector = TraceCollector()

# Simulate a span (timed operation)
collector.start_span("search_op")

import time
time.sleep(0.1)  # Simulate work

event = collector.end_span(
    span_id="search_op",
    session_id="demo_session",
    iteration=1,
    layer="tool",
    operation="semantic_search",
    input_summary="Query: benefits of vector databases",
    output_summary="Found 5 relevant chunks",
    tokens_used=0,
    success=True,
)

print(f"Recorded trace event:")
print(f"  Layer: {event.layer}")
print(f"  Operation: {event.operation}")
print(f"  Latency: {event.latency_ms}ms")
print(f"  Success: {event.success}")

In [ ]:
# Record more events for a complete iteration
events_to_record = [
    ("context", "gather_context", 0, True, None),
    ("tool", "rewrite_query", 150, True, None),
    ("tool", "semantic_search", 0, True, None),
    ("tool", "synthesize_context", 420, True, None),
    ("tool", "draft_report", 1200, True, None),
    ("verification", "quality_score", 380, True, None),
    ("verification", "citation_check", 0, True, None),
    ("verification", "fact_check", 350, False, "missing_citations"),
]

for layer, op, tokens, success, fail_cat in events_to_record:
    collector.record(TraceEvent(
        session_id="demo_session",
        iteration=1,
        layer=layer,
        operation=op,
        tokens_used=tokens,
        latency_ms=50,
        success=success,
        failure_category=fail_cat,
    ))

print(f"Total events recorded: {len(collector.get_events('demo_session'))}")

## 2. Trace Summary

In [ ]:
summary = collector.get_summary("demo_session")
print("Session trace summary:")
print(f"  Total events: {summary['total_events']}")
print(f"  Total tokens: {summary['total_tokens']}")
print(f"  Total latency: {summary['total_latency_ms']}ms")
print(f"  Failures: {summary['failures']}")
print(f"  Failure categories: {summary['failure_categories']}")
print(f"  Events by layer: {summary['events_by_layer']}")

## 3. Persisting Traces to PostgreSQL

In [ ]:
# Persist traces to database
collector.persist("demo_session")
print("Traces persisted to PostgreSQL")

# Query traces from database
import psycopg2

conn = psycopg2.connect(
    host=os.getenv("PGVECTOR_HOST", "localhost"),
    port=os.getenv("PGVECTOR_PORT", "5432"),
    dbname=os.getenv("PGVECTOR_DB", "doc_research"),
    user=os.getenv("PGVECTOR_USER", "postgres"),
    password=os.getenv("PGVECTOR_PASSWORD", "postgres"),
)
cur = conn.cursor()
cur.execute("""
    SELECT layer, operation, tokens_used, latency_ms, success
    FROM trace_events
    WHERE session_id = 'demo_session'
    ORDER BY timestamp
""")

print("\nTraces from PostgreSQL:")
print(f"{'Layer':<15} {'Operation':<20} {'Tokens':<8} {'Latency':<10} {'Success'}")
print("-" * 65)
for row in cur.fetchall():
    print(f"{row[0]:<15} {row[1]:<20} {row[2]:<8} {row[3]:<10} {row[4]}")

cur.close()
conn.close()

## Summary

The trace collector:
- Records every operation with timing and token metrics
- Supports span-based timing (start/end)
- Provides session-level summaries
- Persists to PostgreSQL for historical analysis

Traces form the **Reflect** phase of the harness — enabling the system to learn and improve across iterations.